### Insurance Claims Feature Engineering Pipeline using Snowpark Connect for Apache Spark

This Spark job:
1) Reads raw claims data from External Iceberg tables (FNOL_REPORTS, ADJUSTER_NOTES, INITIAL_ESTIMATES)
   from GLUE_DATABASE_LINKED_DB.INSURANCE_CLAIMS_ICEBERG_GLUE_DB
2) Joins with FRAUD_FLAGS and POLICY_DETAILS tables from Snowflake Native Tables
   in INSURANCE_CLAIMS_INSIGHTS_DB.CLAIMS_ANALYTICS
3) Performs complex feature engineering including:
   - Claim duration metrics
   - Text normalization for adjuster notes
   - Fraud Risk Score calculation using a PRE-TRAINED Python UDF
4) Writes enriched dataset to Snowflake Native Table (CLAIMS_PROCESSED_FEATURES)

Data Flow: Snowflake Tables (Iceberg + Native) -> Spark Processing -> Snowflake Native Table

## 1. Imports and Configuration

In [ ]:
import os
import pickle
import tempfile
from datetime import datetime
from snowflake import snowpark_connect
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, concat, lower, regexp_replace, trim,
    datediff, current_date, coalesce, sum as spark_sum, count,
    udf, split, size, broadcast, current_timestamp, concat_ws, collect_list
)
from pyspark.sql.types import FloatType, StringType, StructType, StructField

In [ ]:
# Output destination
SNOWFLAKE_OUTPUT_DATABASE = "INSURANCE_CLAIMS_INSIGHTS_DB"
SNOWFLAKE_OUTPUT_SCHEMA = "CLAIMS_ANALYTICS"
SNOWFLAKE_OUTPUT_TABLE = "CLAIMS_PROCESSED_FEATURES"

# Source: External Iceberg tables (via Glue catalog)
ICEBERG_DATABASE = "GLUE_DATABASE_LINKED_DB"
ICEBERG_SCHEMA = "insurance_claims_iceberg_glue_db"

# Source: Snowflake Native tables
NATIVE_DATABASE = "INSURANCE_CLAIMS_INSIGHTS_DB"
NATIVE_SCHEMA = "CLAIMS_ANALYTICS"

# Table names
ICEBERG_TABLES = {
    "fnol_reports": "fnol_reports",
    "adjuster_notes": "adjuster_notes",
    "initial_estimates": "initial_estimates",
}

NATIVE_TABLES = {
    "fraud_flags": "FRAUD_FLAGS",
    "policy_details": "POLICY_DETAILS",
}

# Snowflake stage path for the fraud detection model
MODEL_STAGE_PATH = "@INSURANCE_CLAIMS_INSIGHTS_DB.CLAIMS_ANALYTICS.ML_MODELS/fraud_detection_model.pkl"
MODEL_SCRIPT_PATH = "@INSURANCE_CLAIMS_INSIGHTS_DB.CLAIMS_ANALYTICS.ML_MODELS/fraud_model.py"

print(f"Iceberg Source: {ICEBERG_DATABASE}.{ICEBERG_SCHEMA}")
print(f"Native Source: {NATIVE_DATABASE}.{NATIVE_SCHEMA}")
print(f"Output: {SNOWFLAKE_OUTPUT_DATABASE}.{SNOWFLAKE_OUTPUT_SCHEMA}.{SNOWFLAKE_OUTPUT_TABLE}")

## 2. Initialize Spark Session

In [ ]:
def init_spark_session():
    """Initialize Snowpark Connect for Spark session"""
    try:
        spark = snowpark_connect.server.init_spark_session()
        print("Connected to Snowpark Connect for Apache Spark")
        return spark, True
    except ImportError:
        spark = SparkSession.builder \
            .appName("InsuranceClaimsFeatureEngineering") \
            .config("spark.sql.adaptive.enabled", "true") \
            .getOrCreate()
        print("Using local Spark session (Snowpark Connect not available)")
        return spark, False

spark, is_snowpark_connect = init_spark_session()

# Use lowercase table identifiers (Glue default)
spark.conf.set("snowpark.connect.sql.identifiers.auto-uppercase", "none")
# set the model path in spark config
spark.conf.set(
    "snowpark.connect.udf.imports",
    "[@INSURANCE_CLAIMS_INSIGHTS_DB.CLAIMS_ANALYTICS.ML_MODELS/fraud_model.py, @INSURANCE_CLAIMS_INSIGHTS_DB.CLAIMS_ANALYTICS.ML_MODELS/fraud_detection_model.pkl]"
)
spark.conf.set("snowpark.connect.udf.packages", "numpy")

## 3. Load Fraud Detection Model from Stage

In [ ]:
_cached_model = None

def get_fraud_model():
    """Load and cache the pre-trained fraud detection model from Snowflake stage."""
    global _cached_model
    if _cached_model is None:
        from fraud_model import FraudDetectionModel
        try:
            from snowflake.snowpark import Session
            
            print(f"Loading model from Snowflake stage: {MODEL_STAGE_PATH}")
            session = Session.builder.getOrCreate()
            
            with tempfile.TemporaryDirectory() as tmp_dir:
                session.file.get(MODEL_STAGE_PATH, tmp_dir)
                model_filename = os.path.basename(MODEL_STAGE_PATH)
                local_model_path = os.path.join(tmp_dir, model_filename)
                _cached_model = FraudDetectionModel.load(local_model_path)
            
            print("Successfully loaded model from stage")
            
        except Exception as e:
            print(f"Could not load model from stage ({e}), creating new model...")
            _cached_model = FraudDetectionModel()
    
    return _cached_model

# Initialize model
print("Initializing pre-trained fraud detection model...")
get_fraud_model()

## 4. Load Data from Snowflake Tables

In [ ]:
def normalize_column_names(df):
    """Normalize column names to lowercase for consistent processing."""
    for column in df.columns:
        df = df.withColumnRenamed(column, column.lower())
    return df

def read_table(database, schema, table_name):
    """Read a single table from Snowflake"""
    full_table = f"{database}.{schema}.{table_name}"
    return spark.read.table(full_table)

In [ ]:
# Load External Iceberg tables from GLUE_LINKED_DB.insurance_claims_iceberg_db
print(f"\nLoading External Iceberg tables from: {ICEBERG_DATABASE}.{ICEBERG_SCHEMA}")

fnol_reports = read_table(ICEBERG_DATABASE, ICEBERG_SCHEMA, ICEBERG_TABLES["fnol_reports"])
print(f"  - {ICEBERG_TABLES['fnol_reports']}: {fnol_reports.count()} records")

adjuster_notes = read_table(ICEBERG_DATABASE, ICEBERG_SCHEMA, ICEBERG_TABLES["adjuster_notes"])
print(f"  - {ICEBERG_TABLES['adjuster_notes']}: {adjuster_notes.count()} records")

initial_estimates = read_table(ICEBERG_DATABASE, ICEBERG_SCHEMA, ICEBERG_TABLES["initial_estimates"])
print(f"  - {ICEBERG_TABLES['initial_estimates']}: {initial_estimates.count()} records")

In [ ]:
# Load Snowflake Native tables from INSURANCE_CLAIMS_DB.CLAIMS_ANALYTICS
print(f"\nLoading Snowflake Native tables from: {NATIVE_DATABASE}.{NATIVE_SCHEMA}")

fraud_flags = read_table(NATIVE_DATABASE, NATIVE_SCHEMA, NATIVE_TABLES["fraud_flags"])
fraud_flags = normalize_column_names(fraud_flags)
print(f"  - {NATIVE_TABLES['fraud_flags']}: {fraud_flags.count()} records")

policy_details = read_table(NATIVE_DATABASE, NATIVE_SCHEMA, NATIVE_TABLES["policy_details"])
policy_details = normalize_column_names(policy_details)
print(f"  - {NATIVE_TABLES['policy_details']}: {policy_details.count()} records")

## 5. Define UDFs for Fraud Detection

In [ ]:
def create_pretrained_fraud_udf():
    """Create a UDF that uses the pre-trained fraud detection model."""
    
    def predict_fraud_score(
        loss_type, estimate_amount, siu_referral, investigation_status,
        prior_claims_count, historical_risk_score, policy_age_days,
        injuries_reported, reporting_delay_days, estimate_premium_ratio,
        has_fraud_keywords
    ):
        """Apply pre-trained model to predict fraud risk score"""
        model = get_fraud_model()
        
        try:
            score = model.predict_score(
                loss_type=loss_type,
                estimate_amount=float(estimate_amount) if estimate_amount else None,
                siu_referral=siu_referral,
                investigation_status=investigation_status,
                prior_claims_count=int(prior_claims_count) if prior_claims_count else 0,
                historical_risk_score=float(historical_risk_score) if historical_risk_score else None,
                policy_age_days=int(policy_age_days) if policy_age_days else None,
                injuries_reported=injuries_reported,
                reporting_delay_days=int(reporting_delay_days) if reporting_delay_days else None,
                estimate_premium_ratio=float(estimate_premium_ratio) if estimate_premium_ratio else None,
                has_fraud_keywords=bool(has_fraud_keywords) if has_fraud_keywords is not None else False
            )
            return float(score)
        except Exception as e:
            print(f"Error in fraud prediction: {e}")
            return 0.0
    
    return udf(predict_fraud_score, FloatType())


def create_risk_category_udf():
    """Create UDF to categorize fraud risk based on model thresholds"""
    
    def get_risk_category(score):
        """Get risk category from pre-trained model thresholds"""
        if score is None:
            return "Unknown"
        model = get_fraud_model()
        return model.get_risk_category(score)
    
    return udf(get_risk_category, StringType())


fraud_score_udf = create_pretrained_fraud_udf()
risk_category_udf = create_risk_category_udf()
print("UDFs created successfully")

## 6. Aggregate Adjuster Notes

In [ ]:

def normalize_text(text_col):
    """Normalize text by removing special characters and converting to lowercase"""
    return trim(
        regexp_replace(
            regexp_replace(
                lower(text_col),
                r'[^a-z0-9\s]', ' '
            ),
            r'\s+', ' '
        )
    )

def aggregate_adjuster_notes(adjuster_notes):
    """Aggregate adjuster notes per claim with text features"""
    return adjuster_notes \
        .withColumn("normalized_content", normalize_text(col("note_content"))) \
        .groupBy("claim_id") \
        .agg(
            count("*").alias("total_notes_count"),
            spark_sum(when(col("note_type") == "SIU Referral", 1).otherwise(0)).alias("siu_notes_count"),
            spark_sum(when(col("note_type") == "Medical Update", 1).otherwise(0)).alias("medical_notes_count"),
            spark_sum(when(col("note_type") == "Investigation Started", 1).otherwise(0)).alias("investigation_notes_count"),
            concat_ws(" ", collect_list("normalized_content")).alias("combined_notes_sample")
        )

adjuster_notes_agg = aggregate_adjuster_notes(adjuster_notes)
print(f"Aggregated adjuster notes: {adjuster_notes_agg.count()} claims")

## 7. Join All Data Sources

In [ ]:
def join_all_data(fnol_reports, adjuster_notes_agg, initial_estimates, fraud_flags, policy_details):
    """Join all data sources together"""
    print("\nJoining data sources...")
    
    enriched = fnol_reports \
        .join(initial_estimates.select(
            "claim_id",
            col("total_estimate").alias("estimate_amount"),
            col("status").alias("estimate_status"),
            col("estimate_date"),
            col("labor_hours"),
            col("parts_total"),
            col("net_payable")
        ), on="claim_id", how="left") \
        .join(adjuster_notes_agg, on="claim_id", how="left") \
        .join(fraud_flags.select(
            "claim_id",
            col("flag_type"),
            col("flag_source"),
            col("risk_score").alias("fraud_risk_score_historical"),
            col("siu_referral"),
            col("siu_case_number"),
            col("investigation_status"),
            col("investigation_outcome"),
            col("red_flags_identified"),
            col("prior_claims_count")
        ), on="claim_id", how="left") \
        .join(broadcast(policy_details.select(
            "policy_number",
            col("policy_type"),
            col("effective_date").alias("policy_effective_date"),
            col("expiration_date").alias("policy_expiration_date"),
            col("coverage_liability_limit"),
            col("coverage_collision_deductible"),
            col("coverage_comprehensive_deductible"),
            col("annual_premium"),
            col("risk_tier"),
            col("underwriting_score"),
            col("discount_good_driver"),
            col("discount_multi_policy"),
            col("prior_claims_3yr")
        )), on="policy_number", how="left")
    
    print(f"  - Joined dataset: {enriched.count()} records")
    return enriched


joined_df = join_all_data(fnol_reports, adjuster_notes_agg, initial_estimates, fraud_flags, policy_details)

## 8. Feature Engineering

In [ ]:
def calculate_duration_metrics(df):
    """Calculate claim duration and timing metrics"""
    print("\nCalculating duration metrics...")
    
    return df \
        .withColumn(
            "days_to_report",
            datediff(col("date_reported"), col("date_of_loss"))
        ) \
        .withColumn(
            "days_since_loss",
            datediff(current_date(), col("date_of_loss"))
        ) \
        .withColumn(
            "days_to_estimate",
            datediff(col("estimate_date"), col("date_of_loss"))
        ) \
        .withColumn(
            "policy_age_at_loss",
            datediff(col("date_of_loss"), col("policy_effective_date"))
        ) \
        .withColumn(
            "policy_days_remaining",
            datediff(col("policy_expiration_date"), current_date())
        ) \
        .withColumn(
            "reporting_delay_category",
            when(col("days_to_report") == 0, "Same Day")
            .when(col("days_to_report") == 1, "Next Day")
            .when(col("days_to_report") <= 3, "Within 3 Days")
            .when(col("days_to_report") <= 7, "Within Week")
            .otherwise("Delayed")
        )


df_with_duration = calculate_duration_metrics(joined_df)

In [ ]:
def calculate_risk_features(df):
    """Calculate risk-related features"""
    print("Calculating risk features...")
    
    return df \
        .withColumn(
            "loss_type_category",
            when(col("loss_type").isin("Theft", "Fire"), "High Risk")
            .when(col("loss_type").isin("Vandalism", "Hit and Run"), "Medium Risk")
            .otherwise("Standard Risk")
        ) \
        .withColumn(
            "estimate_to_premium_ratio",
            when(col("annual_premium") > 0,
                 col("estimate_amount") / col("annual_premium")
            ).otherwise(lit(None))
        ) \
        .withColumn(
            "has_siu_referral",
            when(col("siu_referral") == "Yes", True).otherwise(False)
        ) \
        .withColumn(
            "has_open_investigation",
            when(col("investigation_status").isin("Open", "Pending", "Monitoring"), True)
            .otherwise(False)
        ) \
        .withColumn(
            "complexity_score",
            (
                when(col("injuries_reported") == "Yes", 20).otherwise(0) +
                when(col("other_party_involved") == "Yes", 15).otherwise(0) +
                when(col("police_report_filed") == "Yes", 10).otherwise(0) +
                when(col("tow_required") == "Yes", 10).otherwise(0) +
                when(col("loss_type").isin("Theft", "Fire"), 25).otherwise(5) +
                coalesce(col("total_notes_count"), lit(0)) * 5
            )
        ) \
        .withColumn(
            "risk_tier_numeric",
            when(col("risk_tier") == "Non-Standard", 4)
            .when(col("risk_tier") == "Standard", 3)
            .when(col("risk_tier") == "Preferred", 2)
            .when(col("risk_tier") == "Preferred Plus", 1)
            .otherwise(3)
        )


df_with_risk = calculate_risk_features(df_with_duration)

In [ ]:
def add_text_features(df):
    """Add text-based features from adjuster notes"""
    print("Adding text-based features...")
    
    return df \
        .withColumn(
            "notes_word_count",
            when(col("combined_notes_sample").isNotNull(),
                 size(split(col("combined_notes_sample"), " "))
            ).otherwise(0)
        ) \
        .withColumn(
            "has_attorney_mention",
            when(col("combined_notes_sample").contains("attorney"), True)
            .otherwise(False)
        ) \
        .withColumn(
            "has_fraud_keywords",
            when(
                col("combined_notes_sample").contains("fraud") |
                col("combined_notes_sample").contains("suspicious") |
                col("combined_notes_sample").contains("investigation") |
                col("combined_notes_sample").contains("inconsisten"),
                True
            ).otherwise(False)
        )


df_with_text = add_text_features(df_with_risk)

## 9. Apply Fraud Detection Model

In [ ]:
def apply_pretrained_fraud_model(df, fraud_score_udf, risk_category_udf):
    """Apply the pre-trained fraud detection model via UDF."""
    print("\nApplying PRE-TRAINED fraud detection model...")
    print(f"  Model path: {MODEL_STAGE_PATH}")
    
    df_with_score = df.withColumn(
        "ml_fraud_risk_score",
        fraud_score_udf(
            col("loss_type"),
            col("estimate_amount"),
            col("siu_referral"),
            col("investigation_status"),
            coalesce(col("prior_claims_count"), col("prior_claims_3yr"), lit(0)),
            col("fraud_risk_score_historical"),
            col("policy_age_at_loss"),
            col("injuries_reported"),
            col("days_to_report"),
            col("estimate_to_premium_ratio"),
            col("has_fraud_keywords")
        )
    )
    
    df_with_category = df_with_score.withColumn(
        "fraud_risk_category",
        risk_category_udf(col("ml_fraud_risk_score"))
    )
    
    return df_with_category


df_with_fraud = apply_pretrained_fraud_model(df_with_text, fraud_score_udf, risk_category_udf)

## 10. Select Final Columns

In [ ]:
def select_final_columns(df):
    """Select and order final output columns optimized for reporting and AI/ML"""
    return df.select(
        col("claim_id").alias("CLAIM_ID"),
        col("policy_number").alias("POLICY_NUMBER"),
        col("date_of_loss").alias("DATE_OF_LOSS"),
        col("date_reported").alias("DATE_REPORTED"),
        col("loss_type").alias("LOSS_TYPE"),
        col("loss_type_category").alias("LOSS_TYPE_CATEGORY"),
        col("claimant_name").alias("CLAIMANT_NAME"),
        col("loss_location_city").alias("LOSS_LOCATION_CITY"),
        col("loss_location_state").alias("LOSS_LOCATION_STATE"),
        col("vehicle_year").alias("VEHICLE_YEAR"),
        col("vehicle_make").alias("VEHICLE_MAKE"),
        col("vehicle_model").alias("VEHICLE_MODEL"),
        col("injuries_reported").alias("INJURIES_REPORTED"),
        col("police_report_filed").alias("POLICE_REPORT_FILED"),
        col("other_party_involved").alias("OTHER_PARTY_INVOLVED"),
        col("tow_required").alias("TOW_REQUIRED"),
        col("estimate_amount").alias("ESTIMATE_AMOUNT"),
        col("estimate_status").alias("ESTIMATE_STATUS"),
        col("net_payable").alias("NET_PAYABLE"),
        col("labor_hours").alias("LABOR_HOURS"),
        col("parts_total").alias("PARTS_TOTAL"),
        col("days_to_report").alias("DAYS_TO_REPORT"),
        col("days_since_loss").alias("DAYS_SINCE_LOSS"),
        col("days_to_estimate").alias("DAYS_TO_ESTIMATE"),
        col("reporting_delay_category").alias("REPORTING_DELAY_CATEGORY"),
        col("total_notes_count").alias("TOTAL_NOTES_COUNT"),
        col("siu_notes_count").alias("SIU_NOTES_COUNT"),
        col("medical_notes_count").alias("MEDICAL_NOTES_COUNT"),
        col("notes_word_count").alias("NOTES_WORD_COUNT"),
        col("has_attorney_mention").alias("HAS_ATTORNEY_MENTION"),
        col("has_fraud_keywords").alias("HAS_FRAUD_KEYWORDS"),
        col("policy_type").alias("POLICY_TYPE"),
        col("policy_age_at_loss").alias("POLICY_AGE_AT_LOSS"),
        col("policy_days_remaining").alias("POLICY_DAYS_REMAINING"),
        col("annual_premium").alias("ANNUAL_PREMIUM"),
        col("risk_tier").alias("RISK_TIER"),
        col("risk_tier_numeric").alias("RISK_TIER_NUMERIC"),
        col("underwriting_score").alias("UNDERWRITING_SCORE"),
        col("coverage_liability_limit").alias("COVERAGE_LIABILITY_LIMIT"),
        col("discount_good_driver").alias("DISCOUNT_GOOD_DRIVER"),
        col("discount_multi_policy").alias("DISCOUNT_MULTI_POLICY"),
        col("prior_claims_3yr").alias("PRIOR_CLAIMS_3YR"),
        col("has_siu_referral").alias("HAS_SIU_REFERRAL"),
        col("siu_case_number").alias("SIU_CASE_NUMBER"),
        col("investigation_status").alias("INVESTIGATION_STATUS"),
        col("investigation_outcome").alias("INVESTIGATION_OUTCOME"),
        col("fraud_risk_score_historical").alias("FRAUD_RISK_SCORE_HISTORICAL"),
        col("estimate_to_premium_ratio").alias("ESTIMATE_TO_PREMIUM_RATIO"),
        col("complexity_score").alias("COMPLEXITY_SCORE"),
        col("ml_fraud_risk_score").alias("ML_FRAUD_RISK_SCORE"),
        col("fraud_risk_category").alias("FRAUD_RISK_CATEGORY"),
        current_timestamp().alias("PROCESSED_AT")
    )


final_df = select_final_columns(df_with_fraud)
print(f"Final dataset has {len(final_df.columns)} columns")

## 11. Summary Statistics

In [ ]:
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print(f"\nTotal Claims Processed: {final_df.count()}")

In [ ]:
print("\n--- Fraud Risk Distribution (Pre-trained Model) ---")
final_df.groupBy("FRAUD_RISK_CATEGORY").count().orderBy("count").show()

In [ ]:
print("\n--- Loss Type Categories ---")
final_df.groupBy("LOSS_TYPE_CATEGORY").count().orderBy("count").show()

In [ ]:
print("\n--- Reporting Delay Distribution ---")
final_df.groupBy("REPORTING_DELAY_CATEGORY").count().orderBy("count").show()

In [ ]:
print("\n--- Risk Tier Distribution ---")
final_df.groupBy("RISK_TIER").count().orderBy("count").show()

In [ ]:
print("\n--- Key Numeric Feature Statistics ---")
final_df.select(
    "ESTIMATE_AMOUNT",
    "ML_FRAUD_RISK_SCORE",
    "COMPLEXITY_SCORE",
    "DAYS_TO_REPORT",
    "POLICY_AGE_AT_LOSS"
).describe().show()

In [ ]:
print("\n--- Top 10 Highest Fraud Risk Claims ---")
final_df.select(
    "CLAIM_ID", "LOSS_TYPE", "ESTIMATE_AMOUNT", 
    "ML_FRAUD_RISK_SCORE", "FRAUD_RISK_CATEGORY"
).orderBy(col("ML_FRAUD_RISK_SCORE").desc()).show(10)

## 12. Write to Snowflake

In [ ]:
full_table_name = f"{SNOWFLAKE_OUTPUT_DATABASE}.{SNOWFLAKE_OUTPUT_SCHEMA}.{SNOWFLAKE_OUTPUT_TABLE}"

print("=" * 60)
print("WRITING TO SNOWFLAKE NATIVE TABLE")
print("=" * 60)
print(f"  Target: {full_table_name}")

final_df.write \
    .format("snowflake") \
    .mode("overwrite") \
    .option("dbtable", SNOWFLAKE_OUTPUT_TABLE) \
    .option("sfDatabase", SNOWFLAKE_OUTPUT_DATABASE) \
    .option("sfSchema", SNOWFLAKE_OUTPUT_SCHEMA) \
    .save()

print(f"  Successfully wrote {final_df.count()} records to {full_table_name}")